# Notebook 05 — Delta Hedging Simulation

The backtest in Notebook 04 left the short straddle **naked** — fully exposed to directional moves.  
In practice, professional options desks **delta hedge** their positions daily to remove directional risk.

**The idea:**  
Every day, compute the net delta of your option position. Buy or sell the underlying index  
to make the total portfolio delta zero. Now you are only exposed to vol and time — not direction.

**What this notebook shows:**
1. What delta hedging is and why we do it
2. Simulate selling a short straddle and hedging daily
3. Track option P&L, hedge P&L, and total P&L
4. Show the cost of hedging (transaction costs eat into the premium collected)
5. Compare hedged vs unhedged performance on the same trade

**Key insight:** delta hedging removes directional risk but *costs* you gamma.  
The hedger's P&L comes purely from the difference between IV and RV — the vol risk premium.

In [ ]:
import sys
sys.path.insert(0, "../src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

from option_pricing import bs_price, implied_volatility
from greeks import delta, gamma, theta, vega

np.random.seed(7)
print("Imports OK")

---
## 1. What is delta?

**Delta** measures how much the option price moves per ₹1 move in spot.

- Long call delta: between 0 and +1
- Long put delta:  between -1 and 0
- Short call delta: between -1 and 0  (flipped — you sold it)
- Short put delta:  between 0 and +1  (flipped)

**Net delta of a short straddle:**
```
net_delta = -delta_call - (-delta_put) ... wait, let's be precise:

Short call: position delta = -delta(call)
Short put:  position delta = -delta(put) = -[delta(call) - 1] = 1 - delta(call)

Net = -delta(call) + (1 - delta(call)) = 1 - 2*delta(call)
```

At ATM: delta(call) ≈ 0.5, so net delta ≈ 0. The straddle starts delta-neutral.  
As spot moves away from strike, delta drifts — that's what we hedge.

In [ ]:
# Visualise how straddle net delta changes with spot
import numpy as np
spots = np.linspace(18_500, 20_500, 300)
K = 19_500
r, sigma, T = 0.065, 0.15, 0.25

net_deltas = []
for s in spots:
    d_call = delta(s, K, r, sigma, T, "call")
    d_put  = delta(s, K, r, sigma, T, "put")
    # Short straddle: -1 call, -1 put
    net = -d_call - d_put   # both legs short
    net_deltas.append(net)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(spots, net_deltas, color="steelblue", lw=2.5)
ax.axhline(0, color="gray", lw=1, ls="--")
ax.axvline(K, color="gray", lw=1, ls="--", label=f"Strike = {K:,}")
ax.set_title("Short Straddle — Net Delta vs Spot", fontsize=13, fontweight="bold")
ax.set_xlabel("Spot"); ax.set_ylabel("Net Delta")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()
print("At ATM: net delta ~ 0")
print("As spot rises: delta turns negative (you're short the rally)")
print("As spot falls: delta turns positive (you're short the drop)")
print("Hedging means buying/selling spot to keep this line near zero every day.")

---
## 2. Set up the simulation

We simulate one short straddle trade across a single expiry cycle (30 days).

**Parameters:**
- Sell 1 lot of ATM straddle at entry
- Hedge delta daily using the underlying (simulate buying/selling NIFTY futures at spot)
- Transaction cost on hedge: 0.05% of notional per rebalance (futures slippage)

**P&L accounting:**
- Option P&L = change in value of the short straddle position
- Hedge P&L  = cumulative profit/loss from the delta hedge trades
- Total P&L  = option P&L + hedge P&L - transaction costs

In [ ]:
# Simulation parameters
S0         = 19_500    # entry spot
K          = 19_500    # ATM strike
r          = 0.065     # risk-free rate
sigma_iv   = 0.16      # implied vol at entry (what we sell)
sigma_rv   = 0.12      # realised vol (what actually happens — lower = we profit)
T_total    = 30        # days to expiry
LOT_SIZE   = 75        # NIFTY lot size
HEDGE_COST = 0.0005    # 0.05% per hedge trade (futures transaction cost)

# Simulate a spot path using realised vol (not IV)
dt       = 1/252
n_steps  = T_total
spot_path = [S0]

for _ in range(n_steps - 1):
    ret = (r - 0.5 * sigma_rv**2) * dt + sigma_rv * np.sqrt(dt) * np.random.normal()
    spot_path.append(spot_path[-1] * np.exp(ret))

dates = pd.date_range("2024-01-02", periods=n_steps, freq="B")
spot_series = pd.Series(spot_path, index=dates)

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(dates, spot_series, color="steelblue", lw=2)
ax.axhline(K, color="gray", ls="--", lw=1.5, label=f"Strike = {K:,}")
ax.fill_between(dates, K, spot_series,
                where=(spot_series > K), alpha=0.15, color="tomato",   label="Above strike")
ax.fill_between(dates, K, spot_series,
                where=(spot_series < K), alpha=0.15, color="steelblue", label="Below strike")
ax.set_title("Simulated NIFTY Spot Path (30 days)", fontsize=12, fontweight="bold")
ax.set_ylabel("Spot"); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()
print(f"Entry spot: {S0:,}   Exit spot: {spot_series.iloc[-1]:,.1f}")

---
## 3. Run the delta hedging loop

Each day:
1. Compute current option values (using IV, since that's what market prices use)
2. Compute net delta of the short straddle
3. Calculate the hedge trade needed to bring delta to zero
4. Record all P&L components

In [ ]:
records = []
hedge_position = 0.0   # current hedge in index units (+ = long, - = short)
cumulative_hedge_pnl  = 0.0
cumulative_hedge_cost = 0.0

for i, (date, spot) in enumerate(spot_series.items()):
    T_remaining = max((T_total - i) / 252, 1e-4)
    
    # Current option values (using IV for pricing — that's what the market shows)
    call_val = bs_price(spot, K, r, sigma_iv, T_remaining, "call")
    put_val  = bs_price(spot, K, r, sigma_iv, T_remaining, "put")
    
    # Short straddle value (we are short both, so position value = -(call + put))
    straddle_val = -(call_val + put_val)
    
    # Option P&L vs entry
    if i == 0:
        entry_call = call_val
        entry_put  = put_val
        entry_straddle_val = straddle_val
    
    option_pnl = (straddle_val - entry_straddle_val) * LOT_SIZE
    
    # Greeks
    d_call = delta(spot, K, r, sigma_iv, T_remaining, "call")
    d_put  = delta(spot, K, r, sigma_iv, T_remaining, "put")
    net_delta = -d_call - d_put   # short both legs
    
    # Hedge trade: buy/sell to neutralise delta
    hedge_trade = -net_delta - hedge_position   # units to trade today
    
    if abs(hedge_trade) > 0.001:
        # Cost of hedge trade
        cost = abs(hedge_trade) * spot * HEDGE_COST * LOT_SIZE
        cumulative_hedge_cost += cost
        
        # P&L from moving the hedge position
        if i > 0:
            prev_spot = spot_series.iloc[i-1]
            cumulative_hedge_pnl += hedge_position * (spot - prev_spot) * LOT_SIZE
        
        hedge_position += hedge_trade
    else:
        if i > 0:
            prev_spot = spot_series.iloc[i-1]
            cumulative_hedge_pnl += hedge_position * (spot - prev_spot) * LOT_SIZE
    
    # Gamma and theta for reference
    g = gamma(spot, K, r, sigma_iv, T_remaining)
    th = theta(spot, K, r, sigma_iv, T_remaining, "call")
    
    records.append({
        "date"           : date,
        "spot"           : round(spot, 2),
        "T_remaining_d"  : round(T_remaining * 252, 1),
        "call_val"       : round(call_val, 2),
        "put_val"        : round(put_val, 2),
        "net_delta"      : round(net_delta, 5),
        "hedge_position" : round(hedge_position, 4),
        "option_pnl"     : round(option_pnl, 2),
        "hedge_pnl"      : round(cumulative_hedge_pnl, 2),
        "hedge_cost"     : round(cumulative_hedge_cost, 2),
        "total_pnl"      : round(option_pnl + cumulative_hedge_pnl - cumulative_hedge_cost, 2),
        "gamma"          : round(g, 6),
        "theta_daily"    : round(th, 4),
    })

sim = pd.DataFrame(records)
print(sim[["date","spot","net_delta","option_pnl","hedge_pnl","total_pnl"]].to_string(index=False))

---
## 4. P&L breakdown chart

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(13, 10), sharex=True)

# Panel 1: Spot vs Strike
axes[0].plot(sim["date"], sim["spot"], color="steelblue", lw=2, label="Spot")
axes[0].axhline(K, color="gray", ls="--", lw=1.5, label=f"Strike {K:,}")
axes[0].set_title("Spot Path", fontweight="bold"); axes[0].set_ylabel("Index")
axes[0].legend(); axes[0].grid(alpha=0.3)
axes[0].yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"{x:,.0f}"))

# Panel 2: P&L components
axes[1].plot(sim["date"], sim["option_pnl"], color="steelblue",  lw=2, label="Option P&L")
axes[1].plot(sim["date"], sim["hedge_pnl"],  color="darkorange", lw=2, label="Hedge P&L")
axes[1].plot(sim["date"], sim["total_pnl"],  color="green",      lw=2.5, ls="--", label="Total P&L")
axes[1].axhline(0, color="gray", lw=0.8)
axes[1].set_title("P&L Components", fontweight="bold"); axes[1].set_ylabel("P&L (Rs)")
axes[1].legend(); axes[1].grid(alpha=0.3)
axes[1].yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"Rs{x:,.0f}"))

# Panel 3: Net delta (how much we needed to hedge)
axes[2].bar(sim["date"], sim["net_delta"], color="steelblue", alpha=0.7, width=0.8)
axes[2].axhline(0, color="gray", lw=0.8)
axes[2].set_title("Net Delta Before Hedge (hedged to 0 each day)", fontweight="bold")
axes[2].set_ylabel("Net Delta"); axes[2].grid(alpha=0.3)

plt.suptitle("Short Straddle — Delta Hedging Simulation", fontsize=13, fontweight="bold")
plt.tight_layout(); plt.show()

---
## 5. Final results

In [ ]:
entry_premium = (entry_call + entry_put) * LOT_SIZE
final = sim.iloc[-1]

# Unhedged P&L (what we would have made with no hedge)
expiry_spot = spot_series.iloc[-1]
call_intrinsic = max(expiry_spot - K, 0)
put_intrinsic  = max(K - expiry_spot, 0)
unhedged_pnl = (entry_call + entry_put - call_intrinsic - put_intrinsic) * LOT_SIZE

print("=" * 50)
print("DELTA HEDGING SIMULATION RESULTS")
print("=" * 50)
print(f"Entry spot          : {S0:,.0f}")
print(f"Exit spot           : {expiry_spot:,.1f}")
print(f"Entry premium (total): Rs{entry_premium:,.2f}")
print()
print(f"Option P&L          : Rs{final['option_pnl']:,.2f}")
print(f"Hedge P&L           : Rs{final['hedge_pnl']:,.2f}")
print(f"Hedge costs         : Rs{final['hedge_cost']:,.2f}")
print(f"Total (hedged) P&L  : Rs{final['total_pnl']:,.2f}")
print()
print(f"Unhedged P&L        : Rs{unhedged_pnl:,.2f}")
print()
print(f"Realised vol used   : {sigma_rv:.0%}")
print(f"Implied vol sold at : {sigma_iv:.0%}")
print(f"IV - RV edge        : {sigma_iv - sigma_rv:.0%}")
print()
if final["total_pnl"] > 0:
    print("Profit — IV > RV, the vol risk premium was captured.")
else:
    print("Loss — RV exceeded IV on this particular path.")

---
## 6. Hedged vs unhedged — multiple paths

A single simulation is noisy. Let's run 200 paths to see the distribution of outcomes.

In [ ]:
N_PATHS = 200
hedged_pnls   = []
unhedged_pnls = []

for trial in range(N_PATHS):
    # New spot path each trial
    path = [S0]
    for _ in range(T_total - 1):
        ret = (r - 0.5*sigma_rv**2)*dt + sigma_rv*np.sqrt(dt)*np.random.normal()
        path.append(path[-1]*np.exp(ret))
    
    # Unhedged P&L
    S_T = path[-1]
    unhedged = (entry_call + entry_put - max(S_T-K,0) - max(K-S_T,0)) * LOT_SIZE
    unhedged_pnls.append(unhedged)
    
    # Hedged P&L (simplified: approximate as IV-RV edge × gamma × notional)
    hedge_pos = 0.0
    cum_h_pnl = 0.0
    cum_cost  = 0.0
    for i in range(1, len(path)):
        T_rem = max((T_total - i)/252, 1e-4)
        d_call = delta(path[i], K, r, sigma_iv, T_rem, "call")
        d_put  = delta(path[i], K, r, sigma_iv, T_rem, "put")
        net_d  = -d_call - d_put
        trade  = -net_d - hedge_pos
        if abs(trade) > 0.001:
            cum_cost += abs(trade) * path[i] * HEDGE_COST * LOT_SIZE
        cum_h_pnl += hedge_pos * (path[i] - path[i-1]) * LOT_SIZE
        hedge_pos += trade
    
    opt_val_T = -(max(S_T-K,0) + max(K-S_T,0))
    opt_pnl = (opt_val_T - entry_straddle_val) * LOT_SIZE
    hedged_pnls.append(opt_pnl + cum_h_pnl - cum_cost)

print(f"Over {N_PATHS} simulated paths:")
print(f"  Hedged   — mean: Rs{np.mean(hedged_pnls):,.0f}   std: Rs{np.std(hedged_pnls):,.0f}   win rate: {np.mean(np.array(hedged_pnls)>0):.0%}")
print(f"  Unhedged — mean: Rs{np.mean(unhedged_pnls):,.0f}   std: Rs{np.std(unhedged_pnls):,.0f}   win rate: {np.mean(np.array(unhedged_pnls)>0):.0%}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

for ax, data, color, label in [
    (ax1, hedged_pnls,   "steelblue",  "Hedged P&L"),
    (ax2, unhedged_pnls, "darkorange", "Unhedged P&L"),
]:
    ax.hist(data, bins=40, color=color, alpha=0.8, edgecolor="white")
    ax.axvline(0,             color="black", lw=1.5)
    ax.axvline(np.mean(data), color="gold",  lw=2, ls="--",
               label=f"Mean: Rs{np.mean(data):,.0f}")
    ax.set_title(label, fontweight="bold")
    ax.set_xlabel("P&L (Rs)"); ax.set_ylabel("Frequency")
    ax.legend(); ax.grid(alpha=0.3)

plt.suptitle(f"P&L Distribution over {N_PATHS} Simulated Paths", fontsize=12, fontweight="bold")
plt.tight_layout(); plt.show()

print()
print("Key observation:")
print("  Hedged P&L has lower variance — directional risk is removed.")
print("  The mean shifts because hedging costs money (transaction costs, bid-ask).")
print("  The remaining P&L is the vol risk premium: IV - RV = {:.0%}".format(sigma_iv - sigma_rv))

---
## Key takeaways

1. **Delta hedging** removes directional risk from an options position.  
   You are no longer betting on which way the market moves — only on vol.

2. **The P&L source** of a delta-hedged short straddle is the IV–RV spread:  
   you sold vol at IV, and the market realised a lower RV. You capture the difference.

3. **Hedging costs money** — each rebalance has transaction costs. Too frequent  
   hedging eats into the edge. In practice, desks hedge when delta exceeds a threshold.

4. **Gamma is your enemy** as a short-vol seller — large spot moves create large  
   deltas that are expensive to hedge. High gamma = high hedging cost.

5. **Theta is your friend** — time decay accrues every day, regardless of the hedge.

---
**All 5 notebooks complete.**  
Next: launch the Streamlit dashboard and prepare the final report.